# MOM 3D Plate - Geometry and Mesh Generation

## Import Packages

In [1]:
try
    using Gmsh: Gmsh, gmsh
catch
    using gmsh
end 

## Section 1: Introduction 

We use [GMSH](gmsh.info) for the following two purposes:

1. the first is to generate the geometry of the 3D metallic plate using the [GMSH OpenCASCADE CAD kernel functions](https://gmsh.info/doc/texinfo/gmsh.html#Namespace-gmsh_002fmodel_002focc). We generate a plate of dimension 1 meter in $x$-direction by 1 meter in $y$-direction by 0.1 meter in $z$-direction.Let us denote this computational domain by $\Omega$. See e.g. [Section Geometry module of Overview of Gmsh](https://gmsh.info/doc/texinfo/gmsh.html#Overview-of-Gmsh) for more details; 

2. the second is to generate a mesh of linear tetrahedral elements (tets) on $\Omega$ using the GMSH mesh functions. We choose the Delaunay mesh generation (the option Mesh.Algorithm3D set to 1) to force the generation of tetrahedral elements only. Let us denote this computational domain by $\Omega^h$. See e.g. [Section Mesh module of Overview of Gmsh](https://gmsh.info/doc/texinfo/gmsh.html#Overview-of-Gmsh) for more details;

The geometry and mesh can be viewed in the GUI of GMSH. 

The mesh can be written to file. 

We use GMSH functionality to read the mesh file and iterate over the elements in the mesh.

## Section 2: Geometry and Mesh Generation 

<b>Exercises</b>
1. change the dimensions of the mesh and generate the mesh again;
2. change the number of elements in the mesh;
3. view the element and node tags in the graphical users interface; 

In [9]:
Gmsh.finalize()

In [18]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.set_number("General.Verbosity",3 )         # make more verbose
#....force the 3D algorithm to Tetrahedra (Default is 1: Delaunay) 4 (Frontal), 7 (MMG3D)
gmsh.option.setNumber("Mesh.Algorithm3D", 1)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature",2)
gmsh.option.setNumber("Mesh.MeshSizeMin",0.2)          # set mesh density 
gmsh.model.mesh.recombine()

#..3/8: generate geometry (unit square 0 <= x <= 1 and 0 <= y <= 1)
volume_tag = gmsh.model.occ.add_box(0,0,0,1,1,0.1,1)

#..4/8: synchronize
gmsh.model.occ.synchronize()

#..5/8: define tag for the boundary 
volumetag = gmsh.model.addPhysicalGroup(3,[1],-1,"Omega")
surfacetag = gmsh.model.addPhysicalGroup(2,[1,2,3,4,5,6],-1,"Gamma")

#..4/8: synchronize
gmsh.model.occ.synchronize()

#..6/8: generate mesh
gmsh.model.mesh.generate(3)

#..7/8: write to file
if (true) gmsh.write("metallic_plate_3d.msh") end
if (false) gmsh.fltk.run() end

#..8/8: finalize
should_finalize && Gmsh.finalize(); 

## Section 3: Read Mesh from File and Iterate over the Elements   

<b>Exercises</b>
1. extend code below to recover faces belonging to an element. In GMSH, this is referred to as the element adjacency. See <i>gmsh.model.mesh.getElementFaceNodes(elementType, faceType, tag = -1, primary = false, task = 0, numTasks = 1)</i>;


In [3]:
?gmsh.model.mesh.getElementFaceNodes()

```
gmsh.model.mesh.getElementFaceNodes(elementType, faceType, tag = -1, primary = false, task = 0, numTasks = 1)
```

Get the nodes on the faces of type `faceType` (3 for triangular faces, 4 for quadrangular faces) of all elements of type `elementType` classified on the entity of tag `tag`. `nodeTags` contains the node tags of the faces for all elements: [e1f1n1, ..., e1f1nFaceType, e1f2n1, ...]. Data is returned by element, with elements in the same order as in `getElements` and `getElementsByType`. If `primary` is set, only the primary (corner) nodes of the faces are returned. If `tag` < 0, get the face nodes for all entities. If `numTasks` > 1, only compute and return the part of the data indexed by `task` (for C++ only; output vector must be preallocated).

Return `nodeTags`.

Types:

  * `elementType`: integer
  * `faceType`: integer
  * `nodeTags`: vector of sizes
  * `tag`: integer
  * `primary`: boolean
  * `task`: size
  * `numTasks`: size


In [11]:
#..1/8: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/8: Read mesh from file
gmsh.open("metallic_plate_3d.msh")

#..3/8: Get the line mesh entity (this part can be skipped) 
# Get all the elementary entities in the model, as a vector of (dimension, tag) pairs
# In case of line tutorial, return dim = 1 and tag = 1 
# or entities[1] = (1,1)
entities = gmsh.model.getEntities(1)

#..4/8: Get all the 3D tetrahedral elements (dim=3) in the plate (tag=1) 
#..we previously made sure to have a single elemType only 
dim = 3; tag = 1
elemTypes, element_tags, node_tags = gmsh.model.mesh.getElements(dim, tag)
nelems = length(element_tags[1])
println(" nelems = " ,nelems)

facenodes = gmsh.model.mesh.getElementFaceNodes(elemTypes[1], 3)
nfacenodes = length(facenodes)
println(" number of facenodes = ", nfacenodes) 

#..5/8: Reshape the flat node_tags array - convenient for later  
num_elements = length(element_tags[1])
nodes_per_element = reshape(node_tags[1], 4, num_elements) # 4 rows, N columns

#..6/8: Get all the nodes in the plate   
nodeTags, node_coord, _ = gmsh.model.mesh.getNodes()

#..7/8: Loop over elements - requires more explanation 
quad = 4
for (i,elemtag) in enumerate(element_tags[1])
    
    #....Extract individual node tags for this tetrahedron
    n0 = nodes_per_element[1, i]
    n1 = nodes_per_element[2, i]
    n2 = nodes_per_element[3, i]
    n3 = nodes_per_element[4, i]
    
    #....Get nodal coordinates 
    # gmsh.model.mesh.getNode returns: (coord, parametricCoord, dim, tag)
    c0, _, _, _ = gmsh.model.mesh.getNode(n0)
    c1, _, _, _ = gmsh.model.mesh.getNode(n1)
    c2, _, _, _ = gmsh.model.mesh.getNode(n2)
    c3, _, _, _ = gmsh.model.mesh.getNode(n3)

 
    if(false)
      println("  elemtag = ",elemtag)  
      println("    elemnode0tag  = ",n0) 
      println("      coord node0 = ",c0)
      println("    elemnode1tag  = ",n1) 
      println("      coord node1 = ",c1)
      println("    elemnode2tag  = ",n2) 
      println("      coord node2 = ",c2)
      println("    elemnode2tag  = ",n3) 
      println("      coord node4 = ",c3)
    end 
    
end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize() 

Info    : Reading 'metallic_plate_3d.msh'...
Info    : 27 entities
Info    : 108 nodes
Info    : 526 elements
Info    : Done reading 'metallic_plate_3d.msh'
 nelems = 262
 number of facenodes = 3144


false

In [12]:
262*12

3144